In [ ]:
question = "cual es el juego mas jugado"

In [2]:
question = globals().get("question", "")

In [3]:
print("QUESTION (inyectada):", question)


QUESTION (inyectada): cual es el juego mas jugado


In [4]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import pg8000
from openai import OpenAI
import re
import warnings
from IPython.display import display, clear_output

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

warnings.filterwarnings(
    "ignore",
    message="pandas only supports SQLAlchemy connectable",
    category=UserWarning,
)


In [ ]:
conn = pg8000.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME", "postgres"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),

)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Conectado")

In [7]:
def construir_prompt(pregunta):

    return f"""
Eres un experto en SQL para PostgreSQL.

Tabla: juegos

Columnas reales:
- name
- rating
- metacritic
- ratings_count
- reviews_text_count
- released
- added
- genres
- platforms

Reglas:
- SOLO devuelve SQL
- NO expliques nada
- Solo SELECT
- Solo puedes consultar la tabla juegos
- Siempre incluye LIMIT

Búsqueda por nombre (MUY IMPORTANTE):
- Si el usuario pregunta por un juego concreto (ej: "que me puedes decir sobre Super Mario World"), debes buscarlo por la columna name usando ILIKE.
- Usa las palabras importantes del nombre y crea condiciones AND. Ejemplo:
  WHERE name ILIKE '%mario%' AND name ILIKE '%world%'
- Usa LIMIT 5 en este tipo de búsquedas.

Año:
- NO existe la columna year. Si el usuario pregunta por un año (ej. 2000), usa:
  EXTRACT(YEAR FROM released) = 2000
  y filtra released IS NOT NULL.

Interpretaciones:
- "mas jugado/mas usado/mas popular" => mayor added
- "mas antiguo" => released ASC filtrando NULL
- "mas moderno/mas reciente/nuevo" => released DESC filtrando NULL
- Si pregunta "antiguo y moderno" o pide dos extremos, usa UNION ALL con dos SELECT.

Ejemplos:

Pregunta: que me puedes decir sobre Super Mario World
SQL:
SELECT * FROM juegos
WHERE name ILIKE '%mario%'
  AND name ILIKE '%world%'
LIMIT 5;

Pregunta: juegos del año 2000
SQL:
SELECT * FROM juegos
WHERE released IS NOT NULL
  AND EXTRACT(YEAR FROM released) = 2000
LIMIT 10;

Pregunta: juego mas jugado
SQL:
SELECT * FROM juegos
ORDER BY added DESC
LIMIT 1;

Pregunta: juego mas antiguo y mas moderno
SQL:
(SELECT * FROM juegos
 WHERE released IS NOT NULL
 ORDER BY released ASC
 LIMIT 1)
UNION ALL
(SELECT * FROM juegos
 WHERE released IS NOT NULL
 ORDER BY released DESC
 LIMIT 1)
LIMIT 2;

Pregunta:
{pregunta}
"""

In [ ]:
def generar_sql(pregunta):

    prompt = construir_prompt(pregunta)

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()


def limpiar_sql(sql):
    sql = sql.replace("```sql", "")
    sql = sql.replace("```", "")
    sql = sql.strip()
    return sql


def validar_pregunta(pregunta):
    bloqueadas = [
        "clima", "tiempo", "politica", "futbol", "música", "musica",
        "receta", "cocina", "medicina", "banco", "bolsa", "bitcoin"
    ]

    p = pregunta.lower().strip()

    if len(p) < 3:
        return False

    if any(b in p for b in bloqueadas):
        return False

    return True


def validar_sql(sql):

    sql_lower = sql.lower().strip()

    palabras_prohibidas = [
        "delete", "drop", "update", "insert",
        "alter", "truncate", "create"
    ]

    if any(p in sql_lower for p in palabras_prohibidas):
        return False, "Query peligrosa detectada"

    sql_inicio = sql_lower.lstrip("( \n\t")
    if not (sql_inicio.startswith("select") or sql_inicio.startswith("with")):
        return False, " Solo se permiten consultas SELECT"

    if "juegos" not in sql_lower:
        return False, " Solo puedes consultar la tabla juegos"

    if "limit" not in sql_lower:
        return False, " La query debe incluir LIMIT"

    tiene_year = re.search(r"\byear\b", sql_lower) is not None
    usa_extract_year = "extract(year from released)" in sql_lower.replace("\n", " ").replace("\t", " ")

    if tiene_year and not usa_extract_year:
        return False, " La columna 'year' no existe. Usa EXTRACT(YEAR FROM released)"

    return True, "OK"


def _es_peticion_lista(pregunta):
    p = (pregunta or "").lower()
    disparadores = [
        "lista", "listado", "enumer", "top ", "top", "ranking", "mejores",
        "peores", "dame ", "dame", "muestr", "muéstr", "mostrar",
        "cuales", "cuáles", "cual es el top", "cuál es el top"
    ]
    return any(d in p for d in disparadores)


def _formatear_lista(df, max_items=10):
    if df is None or df.empty:
        return "No encontré resultados para esa pregunta"

    df2 = df.copy()

    cols_pref = [
        ("name", "rating"),
        ("name", "metacritic"),
        ("name", "added"),
        ("name", None),
    ]

    name_col = None
    metric_col = None
    for a, b in cols_pref:
        if a in df2.columns and (b is None or b in df2.columns):
            name_col = a
            metric_col = b
            break

    if name_col is None:
        name_col = df2.columns[0]

    if metric_col is not None:
        df2 = df2[[name_col, metric_col]].dropna(subset=[name_col]).copy()
    else:
        df2 = df2[[name_col]].dropna(subset=[name_col]).copy()

    df2 = df2.head(int(max_items))

    lineas = []
    for i, row in enumerate(df2.itertuples(index=False), start=1):
        if metric_col is not None:
            nombre = getattr(row, name_col)
            metrica = getattr(row, metric_col)
            lineas.append(f"{i}. {nombre} ({metric_col}: {metrica})")
        else:
            nombre = getattr(row, name_col)
            lineas.append(f"{i}. {nombre}")

    return "\n".join(lineas)


def generar_respuesta(pregunta, sql, df):
    if _es_peticion_lista(pregunta):
        return _formatear_lista(df, max_items=10)

    datos = df.head(5).to_dict(orient="records")

    prompt = f"""
Eres un experto en videojuegos.

Pregunta del usuario:
{pregunta}

SQL ejecutado:
{sql}

Resultados (máximo 5 filas):
{datos}

Tu tarea:
- Responder en lenguaje natural con contexto, pero corto (1-2 párrafos).
- No inventes datos: usa SOLO los resultados proporcionados.
- Si la pregunta es de 1 juego, habla principalmente del primer resultado.
- Si hay varios resultados, menciona 2-4 como máximo.

MUY IMPORTANTE (evitar relleno):
- NO digas frases tipo "No se proporcionan datos para comparar".
- NO pidas disculpas.
- NO digas "según la base disponible".
- Ve directo a la respuesta.
- Si falta algún dato específico (por ejemplo metacritic), simplemente no lo menciones.

Guía de contenido (usa lo que exista en los resultados):
- name
- released
- genres
- rating / metacritic
- added

Respuesta:
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()


def extraer_keywords_busqueda(pregunta):
    p = pregunta.lower()
    for ch in ["?", "¿", "!", "¡", ",", ".", ":", ";", "\"", "'", "(", ")"]:
        p = p.replace(ch, " ")

    stop = {
        "que", "qué", "me", "puedes", "decir", "sobre", "del", "de", "el", "la", "los", "las",
        "un", "una", "juego", "juegos", "dime", "cual", "cuál", "es", "son", "hay", "dame", "lista"
    }

    palabras = [w.strip() for w in p.split() if w.strip()]
    keywords = [w for w in palabras if w not in stop and len(w) >= 3]

    seen = set()
    out = []
    for w in keywords:
        if w not in seen:
            out.append(w)
            seen.add(w)

    return out[:5]


def construir_sql_busqueda_nombre(keywords, modo="and", limit=5):
    if not keywords:
        return None

    conds = [f"name ILIKE '%{k}%'" for k in keywords]

    if modo == "or":
        where = " OR ".join(conds)
    else:
        where = " AND ".join(conds)

    return f"SELECT * FROM juegos WHERE {where} LIMIT {int(limit)};"


def construir_fallbacks_nombre(keywords, limit=5):
    kws = [k for k in keywords if k]
    if not kws:
        return []

    fallbacks = []
    fallbacks.append(construir_sql_busqueda_nombre(kws, modo="and", limit=limit))
    fallbacks.append(construir_sql_busqueda_nombre(kws, modo="or", limit=limit))

    kws2 = kws[:2]
    if len(kws2) >= 2:
        fallbacks.append(construir_sql_busqueda_nombre(kws2, modo="and", limit=limit))
        fallbacks.append(construir_sql_busqueda_nombre(kws2, modo="or", limit=limit))

    kw_principal = "mario" if "mario" in kws else kws[0]
    fallbacks.append(construir_sql_busqueda_nombre([kw_principal], modo="and", limit=limit))

    seen = set()
    out = []
    for q in fallbacks:
        if q and q not in seen:
            out.append(q)
            seen.add(q)
    return out

In [ ]:
def ejecutar_pregunta(pregunta, output=None):
    def _run():
        p = (pregunta or "").strip()

        if not p:
            print("Escribe una pregunta.")
            return

        if not validar_pregunta(p):
            print("Solo puedo responder preguntas relacionadas con videojuegos")
            return

        sql = generar_sql(p)
        sql = limpiar_sql(sql)

        valido, mensaje = validar_sql(sql)
        if not valido:
            print(mensaje)
            print(f"SQL generado: {sql}")
            return

        try:
            df = pd.read_sql(sql, conn)
        except Exception as e:
            print("Error ejecutando el SQL en la base de datos")
            print(f"SQL: {sql}")
            print(f"Error: {e}")
            return

        if df.empty:
            keywords = extraer_keywords_busqueda(p)
            queries_fb = construir_fallbacks_nombre(keywords, limit=5)

            encontrado = False
            for sql_fb in queries_fb:
                try:
                    df_fb = pd.read_sql(sql_fb, conn)
                except Exception:
                    continue

                if not df_fb.empty:
                    sql = sql_fb
                    df = df_fb
                    encontrado = True
                    break

            if not encontrado:
                print("No encontré resultados para esa pregunta")
                return

        respuesta = generar_respuesta(p, sql, df)
        print(respuesta)

    if output is None:
        _run()
    else:
        with output:
            clear_output()
            _run()


pregunta = (question or "").strip()
if pregunta:
    ejecutar_pregunta(pregunta)


El juego más jugado es Grand Theft Auto V, con 22,539 usuarios agregándolo a sus listas. Este título de acción, lanzado en 2013, mantiene una alta popularidad y valoración con un rating de 4.47 basado en 7,360 valoraciones. Está disponible en varias plataformas como PlayStation 5, Xbox Series S/X, PC, y generaciones anteriores de consolas.


In [ ]:
if widgets is not None:
    pregunta_input = widgets.Text(
        placeholder="Ej: ¿Qué me puedes decir sobre Super Mario World?",
        description="Pregunta:",
        layout=widgets.Layout(width="80%"),
    )
    boton = widgets.Button(description="Enviar")
    output = widgets.Output()

    def on_click(b):
        ejecutar_pregunta(pregunta_input.value, output=output)

    boton.on_click(on_click)

    display(pregunta_input, boton, output)
else:
    print("ipywidgets no está disponible en este kernel. Ejecuta con una variable 'question' inyectada o instala ipywidgets.")

Text(value='', description='Pregunta:', layout=Layout(width='80%'), placeholder='Ej: ¿Qué me puedes decir sobr…

Button(description='Enviar', style=ButtonStyle())

Output()